# C1 / C2 受控比较（模型类 / 信息集）

两次单因素对照：**C1** = B4→B5a 只换模型类（特征与样本权重锁死）；**C2** = B5a→B5b 只换信息集（权重与超参锁死）。判定用 ±0.02 nats 四分支（90% 簇区间）。这是实验结果，不是假设。

**v1.4 措辞订正**：第四分支拆成两格——区间**不含 0**（差异可分，只是小于等价边际）标「跨边界，不下结论」；区间**跨 0**（真测不出方向）才标「分辨率不足」。


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd


In [2]:
from src.eval.bootstrap import block_deltas, cluster_bootstrap
from src.eval.decisions import c1_verdict, c2_verdict

def load(arm):
    return pd.read_csv(ROOT / 'results' / 'preds' / f'preds_{arm}.csv',
                       dtype={'match_id': str, 'season': str, 'date': str})

def cmp(name, a, b, vf):
    d = block_deltas(load(a), load(b))   # Δ = LL(a) − LL(b)，正 = b 更好
    ci = cluster_bootstrap(d, B=10_000, alpha=0.10, seed=20261110)
    v = vf(ci['point'], ci['lo'], ci['hi'])
    print(f"{name}: ΔLL({a}) − ΔLL({b}) = {ci['point']:+.4f} nats  "
          f"(90% [{ci['lo']:+.4f}, {ci['hi']:+.4f}], n={d.n_paired})")
    print(f"   → {v['decision']}（{v['reason']}）")

cmp('C1', 'B5a', 'B4', c1_verdict)
cmp('C2', 'B5b', 'B5a', c2_verdict)

C1: ΔLL(B5a) − ΔLL(B4) = +0.0158 nats  (90% [+0.0109, +0.0209], n=16733)
   → 跨边界，不下结论（90% 区间不含 0（差异可分），但整体跨过 ±0.02 等价边际 → 只报点估计与区间，不作实用性宣称）
C2: ΔLL(B5b) − ΔLL(B5a) = -0.0028 nats  (90% [-0.0062, +0.0005], n=16733)
   → 实际等价（90% 区间完全落在 ±0.02 内）


In [3]:
# 主推断表（9 行）里对应行的 log loss 速览
tm = pd.read_csv(ROOT / 'results' / 'table_main.csv').set_index('arm')
print(tm[['log_loss', 'delta_vs_B4']].round(4).to_string())

     log_loss  delta_vs_B4
arm                       
B0     1.0743       0.0840
B1     0.9926       0.0024
B2     0.9855      -0.0048
B3     0.9986       0.0084
B4     0.9903          NaN
B5a    1.0061       0.0158
B5b    1.0033       0.0130
B6     0.9585      -0.0317
B6r    0.9585      -0.0318


**结论**：

- **C1（模型类）**：B4 比 B5a 好 0.0158 nats（90% CI 0.0109–0.0209）。**区间不含 0 ⇒ 差异可分**；但上界 0.0209 越过 +0.02 的等价边际，故按预注册规则**跨边界、不作实用性宣称**（旧措辞「分辨率不足」与「区间不含 0」自相矛盾，v1.4 已改）。读数：同信息集下，透明 Dixon–Coles 与 GBM 的差距小于我们认为有实际意义的量。
- **C2（信息集）**：B5b 比 B5a 好 0.0028 nats（90% CI −0.0062~+0.0005），完全落在 ±0.02 内、**且区间跨 0** → 「实际等价」：加入射门/休整/赛程阶段等全特征既没有带来有意义的提升，也没有带来可分辨的提升。
